<a href="https://colab.research.google.com/github/FDM20000/SIAFAX/blob/main/SIAFAX_YOLO_Unificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SIAFAX - Treinando Yolo

In [13]:
!pip install -q --upgrade ultralytics
from ultralytics import YOLO


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Montando o Google Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!ls /content/drive/MyDrive/projeto_rodovia/Roboflow_zips


rodovia_mvp.v2i-sol-.yolov8.zip  rodovia_mvp.v3-parc_nublado-framesc.yolov8.zip


Extraindo zips do Roboflow >> ensolarado e nublado

In [16]:
!unzip -q "/content/drive/MyDrive/projeto_rodovia/Roboflow_zips/rodovia_mvp.v2i-sol-.yolov8.zip" -d /content/dataset_sun


replace /content/dataset_sun/train/images/frame_000479_jpg.rf.ce91009cd8921f6c9b54a16bd81caae7.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [17]:
!unzip -q "/content/drive/MyDrive/projeto_rodovia/Roboflow_zips/rodovia_mvp.v3-parc_nublado-framesc.yolov8.zip" -d /content/dataset_cloud


replace /content/dataset_cloud/test/images/frame_000534_jpg.rf.b89e784c2be14a0b366729d3c5e773d1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


Verificando estrutura

In [18]:
print("SUN root:")
!ls /content/dataset_sun

print("\nCLOUD root:")
!ls /content/dataset_cloud


SUN root:
data.yaml  README.roboflow.txt	test  train  valid

CLOUD root:
data.yaml  README.roboflow.txt	test  train  valid


UNIFICANDO DATASETS (sun+cloud)

In [19]:
# Criar estrutura base do dataset unificado
!mkdir -p /content/dataset_unificado/train/images
!mkdir -p /content/dataset_unificado/train/labels
!mkdir -p /content/dataset_unificado/valid/images
!mkdir -p /content/dataset_unificado/valid/labels
!mkdir -p /content/dataset_unificado/test/images
!mkdir -p /content/dataset_unificado/test/labels


In [20]:
# Copiar ENSOLARADO
!cp -r /content/dataset_sun/train/images/* /content/dataset_unificado/train/images/
!cp -r /content/dataset_sun/train/labels/* /content/dataset_unificado/train/labels/

!cp -r /content/dataset_sun/valid/images/* /content/dataset_unificado/valid/images/
!cp -r /content/dataset_sun/valid/labels/* /content/dataset_unificado/valid/labels/

!cp -r /content/dataset_sun/test/images/* /content/dataset_unificado/test/images/
!cp -r /content/dataset_sun/test/labels/* /content/dataset_unificado/test/labels/


In [21]:
# Copiar NUBLADO
!cp -r /content/dataset_cloud/train/images/* /content/dataset_unificado/train/images/
!cp -r /content/dataset_cloud/train/labels/* /content/dataset_unificado/train/labels/

!cp -r /content/dataset_cloud/valid/images/* /content/dataset_unificado/valid/images/
!cp -r /content/dataset_cloud/valid/labels/* /content/dataset_unificado/valid/labels/

!cp -r /content/dataset_cloud/test/images/* /content/dataset_unificado/test/images/
!cp -r /content/dataset_cloud/test/labels/* /content/dataset_unificado/test/labels/


Verificação se as cópias foram unificadas

In [22]:
import os

print("Treino - imagens:", len(os.listdir('/content/dataset_unificado/train/images')))
print("Treino - labels:", len(os.listdir('/content/dataset_unificado/train/labels')))

print("Validação - imagens:", len(os.listdir('/content/dataset_unificado/valid/images')))
print("Validação - labels:", len(os.listdir('/content/dataset_unificado/valid/labels')))

print("Teste - imagens:", len(os.listdir('/content/dataset_unificado/test/images')))
print("Teste - labels:", len(os.listdir('/content/dataset_unificado/test/labels')))


Treino - imagens: 2215
Treino - labels: 2215
Validação - imagens: 151
Validação - labels: 151
Teste - imagens: 75
Teste - labels: 75


Criando Dataset unificado yaml

In [23]:
%%writefile /content/data_unificado.yaml
train: /content/dataset_unificado/train/images
val: /content/dataset_unificado/valid/images
test: /content/dataset_unificado/test/images

nc: 9
names:
  0: Aerogerador
  1: Arbusto
  2: Arvore
  3: Caminhao
  4: Carro
  5: Obstaculo
  6: Placa
  7: Poste
  8: Torre


Overwriting /content/data_unificado.yaml


Treinando o Modelo YoLo

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/data_unificado.yaml",
    epochs=100,
    imgsz=960,
    batch=16,
    conf=0.45,
    iou=0.7,
    name="siafax_unificado_v1"
)


Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.45, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_unificado.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=siafax_unificado_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

Salvando treino no Google Drive como ubest.pt (unified best model)

In [ ]:
!cp /content/runs/detect/siafax_unificado_v1/weights/best.pt "/content/drive/MyDrive/projeto_rodovia/ubest.pt"
